
### LangSmith tracing demonstration.
#### *Shows @traceable decorator, metadata tagging, and LLM-as-judge evaluation.*

In [6]:
import os
from dotenv import load_dotenv

from langsmith import traceable, Client
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

langsmith_client = Client()

In [7]:
# ── Basic @traceable usage ────────────────────────────────────────────────────
@traceable(name="fetch_content", tags=["retrieval"])
def fetch_context(query: str) -> str:
    """Simulated retrieval step — traced as its own span."""
    return f"Context for '{query}': PostgreSQL B-tree indexes support equality and range queries."

@traceable(name="generate_answer", tags=["generation"])
def generate_answer(question: str, contect: str) -> str:
    """LLM generation step — traced alongside retrieval."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Answer using only the context provided: {context}"),
        ("human", "{question}"),
    ])
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"context": contect, "question": question})

@traceable(name="full_rag_demo", tags=["rag", "demo"], metadata={"version": "1.0", "enuvironment": "practice"},)
def full_rag_demo(
    question: str,
) -> dict:
    """
    Full RAG pipeline — parent span containing child spans.
    In LangSmith: full_rag_demo contains fetch_context and generate_answer.
    """
    context = fetch_context(question)
    answer = generate_answer(question, context)
    return {"question": question, "answer": answer, "context_used": context}

In [8]:
# ── LLM-as-Judge Evaluation ───────────────────────────────────────────────────
@traceable(name="evaluate_answer_quality")
def evaluate_answer_quality(question: str, answer: str, context: str) -> str:
    """
    Use LLM to evaluate RAG answer quality.
    Three metrics: faithfulness, relevance, completeness.
    """
    eval_prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert evaluator of AI assistant responses.
Score the answer on three dimensions (1-5 each):
- Faithfulness: Does the answer only use information from the context?
- Relevance: Does the answer address the question?
- Completeness: Does the answer fully address the question?

Return JSON only: {{"faithfulness": int, "relevance": int, "completeness": int, "reasoning": str}}"""),
        ("human", """Question: {question}
Context: {context}
Answer: {answer}"""),
    ])

    from langchain_core.output_parsers import JsonOutputParser

    eval_chain = eval_prompt | llm | JsonOutputParser()
    scores = eval_chain.invoke({
        "question": question,
        "context": context,
        "answer": answer,
    })
    return scores

In [9]:
# ── Main demo ─────────────────────────────────────────────────────────────────
def main():
    print("=== LangSmith Tracing Demo ===")
    print("Check smith.langchain.com after running to see traces\n")

    test_questions = [
        "How do PostgreSQL B-tree indexes work?",
        "What is the difference between a composite index and a partial index?",
        "When should I use CREATE INDEX CONCURRENTLY?",
    ]

    for question in test_questions:
        print(f"Q: {question}")
        result = full_rag_demo(question)
        print(f"A: {result['answer'][:100]}...")
        scores = evaluate_answer_quality(
            question=question,
            answer=result["answer"],
            context=result["context_used"],
        )
        print(f"   Faithfulness: {scores.get('faithfulness')}/5 | "
              f"Relevance: {scores.get('relevance')}/5 | "
              f"Completeness: {scores.get('completeness')}/5")
        print()
    
    print("Open smith.langchain.com to see all traces with full details.")

if __name__ == "__main__":
    main()

=== LangSmith Tracing Demo ===
Check smith.langchain.com after running to see traces

Q: How do PostgreSQL B-tree indexes work?
A: PostgreSQL B-tree indexes support equality and range queries....
   Faithfulness: 5/5 | Relevance: 2/5 | Completeness: 1/5

Q: What is the difference between a composite index and a partial index?
A: The provided context doesn’t define composite vs partial indexes. It only says that PostgreSQL B-tre...
   Faithfulness: 5/5 | Relevance: 3/5 | Completeness: 2/5

Q: When should I use CREATE INDEX CONCURRENTLY?
A: The provided context does not specify when to use CREATE INDEX CONCURRENTLY; it only notes that Post...
   Faithfulness: 5/5 | Relevance: 5/5 | Completeness: 5/5

Open smith.langchain.com to see all traces with full details.
